# w9_scale.ipynb — ANCHOR-CAP SCALING SHOWDOWN (ace vs i2ce vs ai2ce)

THE deciding experiment for the scaling thesis: if plain anchored CE (ace)
rises with cap like i2ce does, the "I-CE scales better" claim dies; if ace
saturates/declines while i2ce climbs, it's confirmed. ai2ce rides along to
test whether the anchor rope starts paying at large caps (at 512 it is
worthless: +0.005 ZS). Grid = {ce, i2ce, ai2ce} x {512, 1024, 2048, 4096};
512 all done, i2ce@2048/@4096 done (refs), ce@2048 RESUMES from its ep250
bundle (morning pod died 11:03), everything else fresh. 1024 is a brand-new
cap: the worker builds wscan_gal_rev_g1024.npz on first run.
2048/4096 cells are big-class: run this on an A100 pod (80G for 4096).
Readout is ZS-primary (user protocol), head m4 secondary. AUTO-STOPS.


In [ ]:
# constants
import os

REPO = "/workspace/stable-query-latent"
URL = "https://github.com/Nice9Tian/stable-query-latent.git"
DATA_SRC = "/workspace/fusion_cache_w9"
DATA_RAM = "/dev/shm/fusion_cache_w9"
OUT_DIR = "/workspace/w9_out"          # shared campaign out dir

# the scaling grid (arm, cap) -- done cells skip automatically
FLASH = [
    # cap 1024 (new point) -- small, fastest first
    ("wcle_ce_cetf", 1024),
    ("wcle_i2ce_icetf", 1024),
    ("wcle_ai2ce_icetf", 1024),
    # cap 2048 -- ce resumes ep250; i2ce done (skips)
    ("wcle_ce_cetf", 2048),
    ("wcle_ai2ce_icetf", 2048),
    ("wcle_i2ce_icetf", 2048),
    # cap 4096 -- i2ce done (skips)
    ("wcle_ce_cetf", 4096),
    ("wcle_ai2ce_icetf", 4096),
    ("wcle_i2ce_icetf", 4096),
    # 512 refs (all done, print in readout)
    ("wcle_ce_cetf", 512),
    ("wcle_i2ce_icetf", 512),
    ("wcle_ai2ce_icetf", 512),
]
os.makedirs(OUT_DIR, exist_ok=True)
print(f"{len(FLASH)} scale cells")


In [ ]:
# FORCE-sync repo to origin/main.
import os, importlib.util
if not os.path.isdir(os.path.join(REPO, ".git")):
    !git clone {URL} {REPO}
%cd {REPO}
!git remote set-url origin {URL}
!git fetch origin main
!git reset --hard origin/main
!git rev-parse --short HEAD
for pkg in ("sklearn", "scipy"):
    if importlib.util.find_spec(pkg) is None:
        !pip -q install scikit-learn scipy
        break
import sys
if REPO not in sys.path:
    sys.path.insert(0, REPO)
sys.path.insert(0, os.path.join(REPO, "Pod"))
import w9_jobs as J
print("machinery loaded")


In [ ]:
# Stage the corpus into RAM (llm views not needed).
import shutil
from pathlib import Path
REQUIRED = ["games.npz", "wiki_eval.npz", "wscan_gal_rev.npz",
            "wscan_pool_rev.npy", "wscan_pool_rev_rid.npy", "wscan_pool_rev_len.npy",
            "ss_queries_rev.npz", "ss_queries_rev_S.npy",
            "wiki_clean_views.npz", "sp_raw_views.npz",
            "tag_labels.npz",
            "wiki_eval_split.json", "_tag_splitM.json"]
src = Path(DATA_SRC)
missing = [f for f in REQUIRED if not (src / f).exists()]
assert not missing, f"missing in {DATA_SRC}: {missing}"
dst = Path(DATA_RAM)
dst.mkdir(parents=True, exist_ok=True)
for f in REQUIRED:
    s, d = src / f, dst / f
    if not d.exists() or d.stat().st_size != s.stat().st_size:
        print(f"staging {f} ({s.stat().st_size/1e9:.2f} GB) ...", flush=True)
        shutil.copyfile(s, d)
DATA_DIR = str(dst)
print("corpus in RAM:", DATA_DIR)


In [ ]:
# Full pool: must be READY on the volume; stage onto fast local storage.
import os, time
from pathlib import Path
from Pod.h5_staging import parallel_copy

ready = Path(DATA_SRC) / "full_pool_READY"
assert ready.exists(), "full pool not READY -- run a campaign notebook's build cell once"
src_v = Path(DATA_SRC) / "full_pool_fp16.npy"
src_m = Path(DATA_SRC) / "full_pool_meta.npz"
need = src_v.stat().st_size + (5 << 30)

def _free(p):
    st = os.statvfs(p)
    return st.f_bavail * st.f_frsize

dest_dir = None
for cand in ("/dev/shm", "/root/data", "/root"):
    Path(cand).mkdir(parents=True, exist_ok=True)
    if _free(cand) > need:
        dest_dir = Path(cand)
        break
if dest_dir is None:
    print("WARNING: no local space -- workers will mmap the NETWORK VOLUME copy.")
    FULL_POOL_PATH = str(src_v)
else:
    dst_v = dest_dir / "full_pool_fp16.npy"
    if dst_v.exists() and dst_v.stat().st_size == src_v.stat().st_size:
        print("local full pool already staged:", dst_v)
    else:
        t0 = time.time()
        tmp = dst_v.with_name(dst_v.name + ".copying")
        print(f"staging {src_v.stat().st_size/2**30:.0f} GiB -> {dst_v} ...", flush=True)
        parallel_copy(src_v, tmp, workers=8)
        os.replace(tmp, dst_v)
        print(f"staged in {(time.time()-t0)/60:.1f} min", flush=True)
    import shutil
    shutil.copyfile(src_m, dest_dir / "full_pool_meta.npz")
    FULL_POOL_PATH = str(dst_v)
print("FULL_POOL_PATH =", FULL_POOL_PATH)


In [ ]:
# Drain the matrix across ALL GPUs (heartbeat claims; done cells skip).
import os, queue, subprocess, threading, time
from pathlib import Path

cdir = Path(OUT_DIR) / "claims"
logd = Path(OUT_DIR) / "logs"
logd.mkdir(parents=True, exist_ok=True)
jobs = queue.Queue()
for arm, cap in FLASH:
    nm = J.fs_label(arm, cap, False, 0, "clean", 16)
    if (Path(OUT_DIR) / J.result_name(nm)).exists():
        print(f"[skip] {nm} done"); continue
    jobs.put((arm, cap, nm))
fails = []

def worker(gpu):
    while True:
        try:
            arm, cap, nm = jobs.get_nowait()
        except queue.Empty:
            return
        if not J.try_claim(cdir, nm):
            print(f"[claim] {nm} held elsewhere -- skipped", flush=True)
            continue
        log = logd / f"{arm}_g{cap}.log"
        cmd = ["python", "-u", J.FS_WORKER,
               "--data-dir", DATA_DIR, "--out-dir", OUT_DIR, "--repo", REPO,
               "--arm", arm, "--anchor-cap", str(cap),
               "--epochs", str(J.FS_EPOCHS), "--ckpt-every", str(J.CKPT_EVERY),
               "--ckpt-seeds", str(J.FS_CKPT_SEEDS),
               "--topup-seeds", str(J.TOPUP_SEEDS),
               "--full-pool", "--full-pool-path", FULL_POOL_PATH,
               "--claim-file", str(cdir / f"{nm}.claim")]
        print(f"[gpu{gpu}] start {nm}", flush=True)
        t0 = time.time()
        with open(log, "w") as fh:
            p = subprocess.run(cmd, stdout=fh, stderr=subprocess.STDOUT,
                               env=dict(os.environ, CUDA_VISIBLE_DEVICES=gpu))
        if p.returncode != 0:
            fails.append((nm, str(log)))
        print(f"[gpu{gpu}] " + ("ok" if p.returncode == 0 else "FAIL")
              + f" {nm} [{(time.time()-t0)/60:.1f} min]", flush=True)

stop_evt = threading.Event()
mon = threading.Thread(target=J._monitor, args=([logd], stop_evt), daemon=True)
mon.start()
gpus = J.detect_gpus()
threads = [threading.Thread(target=worker, args=(g,)) for g in gpus]
t0 = time.time()
for t in threads: t.start()
for t in threads: t.join()
stop_evt.set()
print(f"FLASH drained in {(time.time()-t0)/3600:.1f} h; {len(fails)} failed")
for nm, lg in fails:
    print("  FAILED:", nm, "->", lg)


In [ ]:
# Readout: the scaling grid (ZS-primary; head m4 secondary).
import json
import numpy as np
from pathlib import Path
VORD = ["neutral", "noname", "positive", "negative"]
ARMS = [("wcle_ce_cetf", "ce=ace"), ("wcle_i2ce_icetf", "i2ce"),
        ("wcle_ai2ce_icetf", "ai2ce")]
for arm, lab in ARMS:
    for cap in (512, 1024, 2048, 4096):
        nm = f"w9_{arm}" + (f"_g{cap}" if cap != 512 else "")
        zp = Path(OUT_DIR) / f"zs_traj_{nm}_fp.json"
        ft = Path(OUT_DIR) / f"ft4var_{nm}_fp_best.json"
        if not zp.exists():
            print(f"{lab:7s}@{cap:<5} (pending)"); continue
        tr = json.loads(zp.read_text())
        eps = sorted(tr, key=lambda k: int(k[2:]))
        pk = max(eps, key=lambda k: tr[k]["nm_neutral"])
        last = eps[-1]
        line = (f"{lab:7s}@{cap:<5} ZSpeak@{pk[2:]:>4}:"
                f" neu {tr[pk]['nm_neutral']:.3f} non {tr[pk]['nm_noname']:.3f}"
                f" | ep{last[2:]}: {tr[last]['nm_neutral']:.3f}/{tr[last]['nm_noname']:.3f}"
                f" tag(neu/non) {tr[pk]['tag_neutral']:.3f}/{tr[pk]['tag_noname']:.3f}")
        if ft.exists():
            d = json.loads(ft.read_text())
            runs = d["per_seed"]
            m4 = np.mean([np.mean([x[v]["h1"] for x in runs]) for v in VORD])
            line += f" | FT m4 {m4:.3f}@ep{d.get('best_ep')}"
        print(line)
    print()


In [ ]:
# AUTO-STOP the pod (results are on the network volume).
AUTO_STOP = True
if AUTO_STOP:
    import sys
    if REPO not in sys.path:
        sys.path.insert(0, REPO)
    from VICReg_review import pod_selfstop
    pod_id, api_key, ctl = pod_selfstop.preflight("")
    pod_selfstop.stop_pod(pod_id, api_key, ctl)
else:
    print("AUTO_STOP disabled -- stop the pod yourself.")
